# 03 — GSVA statistics

Reads the per-sample master sheet from notebook 02 and tests pathway activity two ways, matching how the DESeq2 figures were built:

- **Longitudinal pairs** (same subjects across visits) → paired Wilcoxon, mirroring the `~ subject + visit` design.
- **vs Healthy** (different subjects) → limma with covariates (sex + age + CMV), mirroring the `~ visit + sex + age + cmv` design.

Both land in one table, `gsva_stats_<tissue>.csv`, with a `test` column saying which produced each row. Run once for `pbmc`, once for `bmmc`.

In [1]:
suppressPackageStartupMessages({
    library(data.table)
    library(dplyr)
    library(tidyr)
    library(tibble)
    library(limma)
  })

## 2. Parameters
The two comparison sets are kept separate because they're tested differently. `healthy_targets` are the visits contrasted against Healthy (limma). `longitudinal_pairs` are the same-subject visit pairs (paired Wilcoxon).

In [2]:
tissue <- "bmmc" # "pbmc" or "bmmc"

scores_path <- sprintf("../../../data/rna/gsva/results/gsva_results/gsva_scores_%s.csv", tissue)
meta_path <- sprintf("../../../manuscript-figures/inputs/metadata/%s_sample_kit_metadata.csv", tissue)
out_dir <- "../../../data/rna/gsva/results/gsva_results"

if (tissue == "pbmc") {
  healthy_targets <- c("PreTx", "PI2C", "EI", "ASCT60d", "ASCT1y", "ASCT2y")
  longitudinal_pairs <- list(
    c("PreTx", "PI2C"), c("PreTx", "EI"), c("PI2C", "EI"),
    c("EI", "ASCT60d"), c("EI", "ASCT1y"), c("EI", "ASCT2y"), c("ASCT1y", "ASCT2y")
  )
} else {
  healthy_targets <- c("PreTx", "EI", "ASCT90d", "ASCT1y", "ASCT2y")
  longitudinal_pairs <- list(
    c("PreTx", "EI"), c("EI", "ASCT90d"), c("EI", "ASCT1y"),
    c("EI", "ASCT2y"), c("ASCT90d", "ASCT1y"), c("ASCT1y", "ASCT2y")
  )
}

## 3. Load scores and attach covariates

  The master sheet is per-sample. We pull sex / age / CMV from the metadata (limma
  needs them) and also build a subject-level table — one value per subject × visit —
  for the paired Wilcoxon track.

In [3]:
gsva_scores <- fread(scores_path)

# sample-level covariates for limma
covars <- read.csv(meta_path) %>%
  distinct(sample.sampleKitGuid, subject.biologicalSex, subject.age, subject.cmv)

gsva_scores <- gsva_scores %>%
  left_join(covars, by = "sample.sampleKitGuid")

# subject-level scores for the paired test (average if a subject has >1 kit at a visit)
gsva_subject <- gsva_scores %>%
  group_by(celltype, pathway, label.visitDetails, subject.subjectGuid) %>%
  summarise(gsva_score = mean(gsva_score), .groups = "drop")

message(
  "samples: ", n_distinct(gsva_scores$sample.sampleKitGuid),
  " | subjects: ", n_distinct(gsva_scores$subject.subjectGuid),
  " | celltypes: ", n_distinct(gsva_scores$celltype),
  " | pathways: ", n_distinct(gsva_scores$pathway)
)

samples: 67 | subjects: 26 | celltypes: 62 | pathways: 1754



> Check: counts look sane (~107 samples for PBMC, ~2,500 → 1,694 pathways, your cell-type count), and gsva_scores$subject.age etc. aren't all-NA.

## 4. Longitudinal track — paired Wilcoxon

For each same-subject visit pair (e.g. PreTx → EI) we take the subjects present at
*both* visits and run a paired Wilcoxon on their subject-level scores. This is the
GSVA analogue of the `~ subject + visit` design — pairing removes each subject's
baseline, which is the inter-patient variability the reviewer worried about.

p-values are left raw here; BH correction happens once at the end (Section 4) so
both tracks are adjusted the same way.

In [4]:
# one paired test for a single celltype × pathway, given two visits
wilcox_paired <- function(d, g1, g2) {
  a <- d %>%
    filter(label.visitDetails == g1) %>%
    select(subject.subjectGuid, x = gsva_score)
  b <- d %>%
    filter(label.visitDetails == g2) %>%
    select(subject.subjectGuid, y = gsva_score)

  m <- inner_join(a, b, by = "subject.subjectGuid") # subjects seen at both visits
  if (nrow(m) < 3) {
    return(tibble()) # too few pairs to test
  }

  tibble(
    test         = "wilcox_paired",
    n_pairs      = nrow(m),
    median1      = median(m$x),
    median2      = median(m$y),
    delta_median = median(m$y) - median(m$x), # +ve = higher at the later visit
    p_value      = wilcox.test(m$x, m$y, paired = TRUE)$p.value
  )
}

In [5]:
wilcox_tbl <- bind_rows(lapply(longitudinal_pairs, function(cmp) {
  g1 <- cmp[1]
  g2 <- cmp[2]
  message("paired: ", g1, " vs ", g2)

  gsva_subject %>%
    filter(label.visitDetails %in% c(g1, g2)) %>%
    group_by(celltype, pathway) %>%
    group_modify(~ wilcox_paired(.x, g1, g2)) %>%
    ungroup() %>%
    mutate(group1 = g1, group2 = g2)
}))

message("paired rows: ", nrow(wilcox_tbl))

paired: PreTx vs EI

paired: EI vs ASCT90d

paired: EI vs ASCT1y

paired: EI vs ASCT2y

paired: ASCT90d vs ASCT1y

paired: ASCT1y vs ASCT2y

paired rows: 542312



>Check: wilcox_tbl has n_pairs values that look like real subject counts (not 1–2), and one block of rows per longitudinal pair.

## 5. vs-Healthy track — limma with covariates

  Healthy donors are different people from the patients, so there's nothing to pair.
  Instead we fit one linear model per cell type on the per-sample scores,
  `~ 0 + visit + sex + age + CMV`, and pull out each "patient visit vs Healthy"
  contrast. Same covariates as the `vs_healthy` DESeq2 runs, so the GSVA supplement
  stays aligned with the main figures.

  We work per cell type (each gets its own model). Cell types with too few samples
  or a degenerate covariate (e.g. only one sex present) are skipped and logged.

In [6]:
run_limma_vs_healthy <- function(ct) {
    tryCatch({
      d <- gsva_scores %>%
        filter(celltype == ct,
               label.visitDetails %in% c("Healthy", healthy_targets),
               !is.na(subject.biologicalSex), !is.na(subject.age), !is.na(subject.cmv))

      if (n_distinct(d$sample.sampleKitGuid) < 6) return(tibble())

      # pathways (rows) × samples (cols)
      mat <- d %>%
        select(sample.sampleKitGuid, pathway, gsva_score) %>%
        pivot_wider(names_from = sample.sampleKitGuid, values_from = gsva_score) %>%
        tibble::column_to_rownames("pathway") %>%
        as.matrix()

      # covariates aligned to the matrix columns
      smeta <- d %>%
        distinct(sample.sampleKitGuid, label.visitDetails,
                 subject.biologicalSex, subject.age, subject.cmv)
      smeta <- smeta[match(colnames(mat), smeta$sample.sampleKitGuid), ]

      visit   <- factor(smeta$label.visitDetails)
      targets <- intersect(healthy_targets, levels(visit))
      if (!("Healthy" %in% levels(visit)) || length(targets) == 0) return(tibble())

      design <- model.matrix(
        ~ 0 + visit + subject.biologicalSex + subject.age + subject.cmv,
        data = smeta
      )
      colnames(design) <- make.names(colnames(design))

      fit <- lmFit(mat, design)
      cm  <- makeContrasts(
        contrasts = paste0("visit", targets, " - visitHealthy"),
        levels    = design
      )
      colnames(cm) <- targets
      fit2 <- eBayes(contrasts.fit(fit, cm))

      bind_rows(lapply(targets, function(t) {
        tt <- topTable(fit2, coef = t, number = Inf, sort.by = "none")
        tibble(
          celltype = ct,
          pathway  = rownames(tt),
          group1   = "Healthy",
          group2   = t,
          test     = "limma_adj",
          logFC    = tt$logFC,
          p_value  = tt$P.Value
        )
      }))
    }, error = function(e) {
      message("limma skipped ", ct, ": ", e$message)
      tibble()
    })
  }

In [7]:
limma_tbl <- bind_rows(lapply(unique(gsva_scores$celltype), run_limma_vs_healthy))
message("limma rows: ", nrow(limma_tbl), " | celltypes: ", n_distinct(limma_tbl$celltype))

limma rows: 457625 | celltypes: 62



> Check: limma rows is non-zero and covers most cell types (a few skips are fine), and logFC sign makes sense for a pathway you know (e.g. IFN higher than Healthy → positive at PreTx).

  ## 6. Combine both tracks and save

  Stack the two tracks, then BH-correct **within each comparison** (so each
  visit-vs-visit or visit-vs-Healthy contrast is corrected across all cell types ×
  pathways — the same scope used in the DESeq2/FGSEA work). The `test` column records
  which method produced each row. This file feeds notebook 04 (plots).

In [8]:
stats_tbl <- bind_rows(wilcox_tbl, limma_tbl) %>%
    group_by(group1, group2) %>%
    mutate(p_adj_BH = p.adjust(p_value, method = "BH")) %>%
    ungroup() %>%
    mutate(p_signif = cut(p_adj_BH,
             breaks = c(-Inf, 1e-4, 1e-3, 1e-2, 5e-2, Inf),
             labels = c("****", "***", "**", "*", "ns")))

  out_file <- file.path(out_dir, sprintf("gsva_stats_%s.csv", tissue))
  fwrite(stats_tbl, out_file)

  message("Saved ", nrow(stats_tbl), " rows -> ", out_file)

Saved 999937 rows -> results/gsva_results/gsva_stats_bmmc.csv



> Check: stats_tbl %>% count(test) shows both wilcox_paired and limma_adj; stats_tbl %>% count(group1, group2, test) shows Healthy comparisons are all limma_adj and longitudinal ones all wilcox_paired.

In [11]:
summary(stats_tbl)

   celltype           pathway              test              n_pairs      
 Length:999937      Length:999937      Length:999937      Min.   : 3      
 Class :character   Class :character   Class :character   1st Qu.: 5      
 Mode  :character   Mode  :character   Mode  :character   Median : 8      
                                                          Mean   : 8      
                                                          3rd Qu.:10      
                                                          Max.   :11      
                                                          NA's   :457625  
    median1          median2        delta_median       p_value      
 Min.   :-0.7     Min.   :-0.7     Min.   :-1.1     Min.   :0.0000  
 1st Qu.:-0.1     1st Qu.:-0.1     1st Qu.:-0.1     1st Qu.:0.1250  
 Median : 0.0     Median : 0.0     Median : 0.0     Median :0.3652  
 Mean   : 0.0     Mean   : 0.0     Mean   : 0.0     Mean   :0.4168  
 3rd Qu.: 0.0     3rd Qu.: 0.1     3rd Qu.: 0.2     3rd